# Phase 7: completed timing-spike diagnostic

Notebook 07 completed on T4 in run `20261024T121852Z-4747d5` at source `da47db9`:
**200 tests passed in the T4 runtime** and all **332 cases** completed (326 history + 3 + 3 controls).
Weights, source contracts and archived sampled-input fingerprints matched.

Four new 251.483–266.993 ms online passes contained 227.667–239.758 ms full Python collections.
The old batch-512 batch-4 repeat-13 HGAM position instead took 28.770 ms with no GC overlap.
This strongly supports collection as an explanation consistent with the old spikes; those
untraced events cannot be proved retroactively. Two other early slow passes had no GC.

Both short automatic target-window controls had no full collections, so they cannot show
that cycle-free hashing fixed the spike. Forced collection moved 88.656 seconds of cleanup
outside executor timers. These diagnostic times do not replace notebook 06's 8.77–8.91%
primary observations. See the [diagnostic findings](https://github.com/WeaversMask/temporal-relational-gnn/blob/main/reports/SPIKE_DIAGNOSTIC_STATUS.md).

The revised [notebook 08](https://github.com/WeaversMask/temporal-relational-gnn/blob/main/notebooks/08_gc_aware_inference.ipynb)
retains natural cleanup with cycle-free auditing. Its two independent sessions
showed 8.38%/8.59% primary savings, with remaining non-GC tails retained.
For deliberate reproduction of this diagnostic:

1. Start a fresh **T4** runtime and disconnect other managed runtimes.
2. Enter the displayed balance and hourly rate in their named fields below.
3. Run the cells in order and upload the **same notebook 05 `training.pt`** when prompted.
4. Download the full results ZIP after success or failure, then disconnect/delete the runtime.

The original-allocation-history condition repeats sizes 32 and 128 before the first six 512
batches. This can take roughly as long as notebook 06; setup and diagnostic overhead vary.
The two control conditions then measure only 512 batches 3–5. Extra tracing can move the
collection point, so examine all observed pauses rather than requiring the exact old position.
The training code, original benchmark files, weights and numerical tolerances remain unchanged.


Public setup leaves `USE_GITHUB_TOKEN = False`. Only for a private historical copy, set it to
`True` in the clone cell and enable a read-only `GITHUB_TOKEN` Colab secret for that repository.
Never paste a token into a cell, output or Git URL.

In [ ]:
import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import tempfile
import uuid
from datetime import datetime, timezone
from pathlib import Path

DEVICE = "cuda"
COMPUTE_UNITS_PER_HOUR = None
COMPUTE_UNITS_REMAINING = None
ISOLATED_RUNTIME_CONFIRMED = False
CHECKPOINT_INPUT = None  # Same notebook 05 training.pt, or None to upload below.
RESERVE_UNITS = 50
PROJECT_ROOT = Path("/content/temporal-relational-gnn")
DATA_DIR = PROJECT_ROOT / "data/rel-stack"
PROJECT_REF = "main"
REPO_URL = "https://github.com/WeaversMask/temporal-relational-gnn.git"

if "_SPIKE_RUN_DIR" not in globals():
    _SPIKE_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6]
    )
    _SPIKE_RUN_DIR = Path("/content/temporary-trgnn-spike-diagnostic/runs") / _SPIKE_RUN_ID
    _SPIKE_RUN_DIR.mkdir(parents=True, exist_ok=False)
run_id = _SPIKE_RUN_ID
RUN_DIR = _SPIKE_RUN_DIR
CHECKPOINT_PATH = RUN_DIR / "checkpoints/training.pt"
TRAINING_REPORT = PROJECT_ROOT / "reports/colab/20261024T073010Z-1f9800/training-run.json"
print("Diagnostic run directory:", RUN_DIR)


def run_logged(arguments, log_name, *, cwd=None, env=None):
    with (RUN_DIR / log_name).open("a") as log:
        with subprocess.Popen(
            arguments,
            cwd=PROJECT_ROOT if cwd is None else cwd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        ) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            result = process.wait()
    if result:
        raise subprocess.CalledProcessError(result, arguments)


def observed_nonnegative_number(name, value):
    if value is not None and (
        isinstance(value, bool)
        or not isinstance(value, (int, float))
        or not math.isfinite(value)
        or value < 0
    ):
        raise ValueError(f"{name} must be an observed non-negative number or None")


observed_nonnegative_number("COMPUTE_UNITS_PER_HOUR", COMPUTE_UNITS_PER_HOUR)
observed_nonnegative_number("COMPUTE_UNITS_REMAINING", COMPUTE_UNITS_REMAINING)
budget_path = RUN_DIR / "colab-budget.json"
if not budget_path.exists():
    budget_path.write_text(
        json.dumps(
            {
                "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
                "units_per_hour_observed": COMPUTE_UNITS_PER_HOUR,
                "units_remaining_observed": COMPUTE_UNITS_REMAINING,
                "isolated_runtime_confirmed": ISOLATED_RUNTIME_CONFIRMED,
                "planned_total_units": 200,
                "reserve_units": RESERVE_UNITS,
                "scope": "current_runtime_interval_only",
                "diagnostic": True,
            },
            indent=2,
            allow_nan=False,
        )
        + "\n"
    )
else:
    print("Preserving the original starting budget observation for this run folder.")
metadata_path = RUN_DIR / "notebook-run.json"
if metadata_path.exists():
    run_metadata = json.loads(metadata_path.read_text())
    if run_metadata["device"] != DEVICE:
        raise ValueError("Keep the same device for this diagnostic run folder")
else:
    run_metadata = {
        "notebook": "07_spike_diagnostic",
        "run_id": run_id,
        "device": DEVICE,
        "project_ref": PROJECT_REF,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    metadata_path.write_text(json.dumps(run_metadata, indent=2) + "\n")
# Pin reruns of this run folder to the commit used by the first successful checkout.
EFFECTIVE_PROJECT_REF = run_metadata.get("source_commit", PROJECT_REF)
print("Project revision for this run:", EFFECTIVE_PROJECT_REF)
if DEVICE not in {"cpu", "cuda"}:
    raise ValueError('Set DEVICE to "cpu" or "cuda"')
if COMPUTE_UNITS_REMAINING is not None and COMPUTE_UNITS_REMAINING <= RESERVE_UNITS:
    raise RuntimeError("The 50-unit reserve has been reached. Export the files and disconnect.")
if COMPUTE_UNITS_PER_HOUR is None or COMPUTE_UNITS_REMAINING is None:
    print("Unavailable budget observations are recorded as null.")

Clone the project and install the same training/runtime dependencies. The run folder retains its source revision on cell reruns.


In [ ]:
# Public cloning needs no Colab secret. Enable only for a private historical copy.
USE_GITHUB_TOKEN = False

token = None
clone_environment = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
for inherited_name in ("GIT_ASKPASS", "SSH_ASKPASS", "TRGNN_CLONE_TOKEN"):
    clone_environment.pop(inherited_name, None)
try:
    with tempfile.TemporaryDirectory() as temporary:
        if USE_GITHUB_TOKEN:
            from google.colab import userdata

            token = userdata.get("GITHUB_TOKEN")
            askpass = Path(temporary) / "askpass.py"
            askpass.write_text(
                "#!/usr/bin/env python3\n"
                "import os, sys\n"
                'print("x-access-token" if "username" in sys.argv[1].lower() '
                'else os.environ["TRGNN_CLONE_TOKEN"])\n'
            )
            askpass.chmod(0o700)
            clone_environment.update({"GIT_ASKPASS": str(askpass), "TRGNN_CLONE_TOKEN": token})
        if not PROJECT_ROOT.exists():
            command = ["git", "-c", "credential.helper=", "clone", REPO_URL, str(PROJECT_ROOT)]
        else:
            command = [
                "git",
                "-C",
                str(PROJECT_ROOT),
                "-c",
                "credential.helper=",
                "fetch",
                "origin",
            ]
        run_logged(command, "clone.log", cwd=Path("/content"), env=clone_environment)
except Exception as exc:
    (RUN_DIR / "setup-failure.json").write_text(
        json.dumps({"stage": "clone", "error_type": type(exc).__name__}, indent=2) + "\n"
    )
    raise
finally:
    token = None
    clone_environment.pop("TRGNN_CLONE_TOKEN", None)
    clone_environment.pop("GIT_ASKPASS", None)

run_logged(["git", "checkout", EFFECTIVE_PROJECT_REF], "checkout.log")
if EFFECTIVE_PROJECT_REF == "main":
    run_logged(["git", "merge", "--ff-only", "origin/main"], "checkout.log")
run_logged(["git", "rev-parse", "HEAD"], "checkout.log")
source_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
if run_metadata.get("source_commit", source_commit) != source_commit:
    raise RuntimeError("This run folder must retain its original source commit")
run_metadata["source_commit"] = source_commit
metadata_path.write_text(json.dumps(run_metadata, indent=2) + "\n")
run_logged(
    [sys.executable, "-m", "pip", "install", "-e", f"{PROJECT_ROOT}[dev,relbench]"],
    "install.log",
)

Check the sampler, GPU and current correctness suite before reconstructing the saved model.


In [ ]:
run_logged([sys.executable, "scripts/install_sampler.py"], "install-sampler.log")
arguments = [
    sys.executable,
    "-m",
    "trgnn.preflight",
    "--check-sampler",
    "--output",
    str(RUN_DIR / "preflight.json"),
]
if DEVICE == "cuda":
    arguments.append("--require-cuda")
run_logged(arguments, "preflight.log")
run_logged([sys.executable, "-m", "pytest", "-q"], "correctness.log")

Verify the diagnostic configuration, original source report and parent training report before loading data.


In [ ]:
import yaml

INFERENCE_CONFIG = json.loads((PROJECT_ROOT / "configs/spike_diagnostic.json").read_text())
frozen_config_path = RUN_DIR / "frozen-diagnostic-config.json"
if frozen_config_path.exists():
    if json.loads(frozen_config_path.read_text()) != INFERENCE_CONFIG:
        raise RuntimeError("This run folder must retain its original frozen configuration")
else:
    frozen_config_path.write_text(json.dumps(INFERENCE_CONFIG, indent=2) + "\n")
print(json.dumps(INFERENCE_CONFIG, indent=2))


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


if sha256_file(TRAINING_REPORT) != INFERENCE_CONFIG["training_report_sha256"]:
    raise RuntimeError("The parent training report must match the frozen report SHA256")
parent_report = json.loads(TRAINING_REPORT.read_text())
if parent_report.get("status") != "passed" or not parent_report.get("trained"):
    raise RuntimeError("The parent must be the successful trained notebook 05 run")
if parent_report["checkpoint"]["sha256"] != INFERENCE_CONFIG["checkpoint_sha256"]:
    raise RuntimeError("The parent checkpoint hash differs from the frozen configuration")

SOURCE_PROFILE = PROJECT_ROOT / INFERENCE_CONFIG["source_profile"]
if sha256_file(SOURCE_PROFILE) != INFERENCE_CONFIG["source_profile_sha256"]:
    raise RuntimeError("The archived reference report changed")
reference_profile = json.loads(SOURCE_PROFILE.read_text())
if reference_profile["benchmark_contract_sha256"] != INFERENCE_CONFIG["benchmark_contract_sha256"]:
    raise RuntimeError("Reference benchmark contract mismatch")

pins = json.loads((PROJECT_ROOT / "configs/upstream.json").read_text())
source_receipt = DATA_DIR / ".trgnn-source.json"
try:
    cached_source = json.loads(source_receipt.read_text())
except (OSError, ValueError):
    cached_source = None
required_inputs = [
    DATA_DIR / "manifest.yaml",
    DATA_DIR / "tasks/user-badge/manifest.yaml",
    DATA_DIR / "tasks/user-badge/train.parquet",
    DATA_DIR / "tasks/user-badge/val.parquet",
]
if (DATA_DIR / "manifest.yaml").is_file():
    database_spec = yaml.safe_load((DATA_DIR / "manifest.yaml").read_text())
    required_inputs.extend(DATA_DIR / "db" / f"{name}.parquet" for name in database_spec["tables"])
NEEDS_DOWNLOAD = cached_source != pins["dataset"] or any(not p.is_file() for p in required_inputs)
print("Pinned training/validation inputs need download:", NEEDS_DOWNLOAD)

Upload the unchanged notebook 05 `training.pt`, exactly as for notebook 06. No new training occurs.


In [ ]:
checkpoint_candidate = CHECKPOINT_INPUT
if checkpoint_candidate is None and CHECKPOINT_PATH.is_file():
    checkpoint_candidate = CHECKPOINT_PATH
if checkpoint_candidate is None:
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Choose the single training.pt checkpoint from notebook 05's ZIP")
    _, checkpoint_bytes = next(iter(uploaded.items()))
    restored = Path("/content/trgnn-inference-import") / run_id / "training.pt"
    restored.parent.mkdir(parents=True, exist_ok=True)
    restored.write_bytes(checkpoint_bytes)
    checkpoint_candidate = restored
    del checkpoint_bytes, uploaded
checkpoint_candidate = Path(checkpoint_candidate)
if not checkpoint_candidate.is_file():
    raise FileNotFoundError(f"No notebook 05 checkpoint at {checkpoint_candidate}")
checkpoint_sha256 = sha256_file(checkpoint_candidate)
if checkpoint_sha256 != INFERENCE_CONFIG["checkpoint_sha256"]:
    raise RuntimeError("Checkpoint SHA256 does not match the reviewed notebook 05 training.pt")
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
if checkpoint_candidate.resolve() != CHECKPOINT_PATH.resolve():
    shutil.copy2(checkpoint_candidate, CHECKPOINT_PATH)
checkpoint_provenance = {
    "sha256": checkpoint_sha256,
    "bytes": CHECKPOINT_PATH.stat().st_size,
    "training_report_sha256": INFERENCE_CONFIG["training_report_sha256"],
    "parent_training_run": INFERENCE_CONFIG["parent_training_run"],
    "use": "fixed_model_spike_diagnostic_only",
}
(RUN_DIR / "checkpoint-input.json").write_text(json.dumps(checkpoint_provenance, indent=2) + "\n")
print("Verified unchanged notebook 05 checkpoint:", CHECKPOINT_PATH)

Run the diagnostic. Each completed case is saved separately, including raw collection events and original timer boundaries. Full validation replay must pass before observation.


In [ ]:
arguments = [
    sys.executable,
    "-m",
    "trgnn.spike_diagnostic",
    "--checkpoint",
    str(CHECKPOINT_PATH),
    "--training-report",
    str(TRAINING_REPORT),
    "--source-profile",
    str(SOURCE_PROFILE),
    "--data-dir",
    str(DATA_DIR),
    "--device",
    DEVICE,
    "--batch-indices",
    *map(str, INFERENCE_CONFIG["batch_indices"]),
    "--output",
    str(RUN_DIR / "spike-diagnostic.json"),
    "--artifacts-dir",
    str(RUN_DIR / "diagnostic-cases"),
]
if NEEDS_DOWNLOAD:
    arguments.append("--download")
if INFERENCE_CONFIG["replay_prefix"]:
    arguments.append("--replay-prefix")
run_logged(arguments, "spike-diagnostic.log")

Inspect completion and save every raw trace. The completed diagnostic aligned full collections with four new slow passes; this supports that mechanism without proving the cause of the untraced historical spikes. Short controls do not establish spike removal or eliminate cleanup cost.


In [ ]:
report_path = RUN_DIR / "spike-diagnostic.json"
if report_path.is_file():
    report = json.loads(report_path.read_text())
    print("Diagnostic status:", report.get("status"))
    print("Completed cases:", len(report.get("cases", [])), "/", report.get("expected_cases", "?"))
    print("Saved prediction replay:", report.get("validation_replay", {}).get("status"))
    print("Causal status:", report.get("causal_status"))
    long_pass_count = 0
    for case in report.get("cases", []):
        summary = case.get("summary", {})
        slow = summary.get("online_passes_over_100_ms", [])
        if slow:
            long_pass_count += len(slow)
            print(
                "\nSlow passes:",
                case["phase"],
                "size",
                case["batch_size"],
                "batch",
                case["batch_index"],
            )
            print(json.dumps(slow, indent=2))
    print("Online passes above 100 ms:", long_pass_count)
    print("Error:", report.get("error", "none"))
    print(
        "A matching collection is evidence about the new observed pause; "
        "historical causality remains unconfirmed."
    )
    print("Keep the complete ZIP including diagnostic-cases and partial traces.")
else:
    print("No diagnostic report yet. Export the available logs and checkpoint.")

Record the current ending balance after the diagnostic finishes or stops.


In [ ]:
ENDING_UNITS_REMAINING = None
observed_nonnegative_number("ENDING_UNITS_REMAINING", ENDING_UNITS_REMAINING)
starting_budget = json.loads((RUN_DIR / "colab-budget.json").read_text())
starting_balance = starting_budget["units_remaining_observed"]
account_delta = (
    round(starting_balance - ENDING_UNITS_REMAINING, 2)
    if starting_balance is not None and ENDING_UNITS_REMAINING is not None
    else None
)
isolated_delta = (
    account_delta
    if starting_budget.get("isolated_runtime_confirmed")
    and account_delta is not None
    and account_delta >= 0
    else None
)
ending_budget = {
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "starting_observation_at_utc": starting_budget.get("recorded_at_utc"),
    "starting_units_remaining_observed": starting_balance,
    "ending_units_remaining_observed": ENDING_UNITS_REMAINING,
    "account_balance_decrease_units": account_delta,
    "balance_delta_precision_decimal_places": 2,
    "isolated_run_consumption_units": isolated_delta,
    "isolated_runtime_confirmed": starting_budget.get("isolated_runtime_confirmed", False),
    "scope": "current_runtime_interval_only",
    "note": (
        "Attribution needs observed balances and an isolated runtime; "
        "earlier intervals are separate."
    ),
}
(RUN_DIR / "budget-end.json").write_text(
    json.dumps(ending_budget, indent=2, allow_nan=False) + "\n"
)
print(json.dumps(ending_budget, indent=2))
if ENDING_UNITS_REMAINING is not None and ENDING_UNITS_REMAINING <= RESERVE_UNITS:
    print("The 50-unit reserve has been reached. Export and disconnect; do not start another run.")

Download the entire diagnostic run folder after success or failure, before disconnecting. The checkpoint is included in your ZIP, not in Git.


In [ ]:
from google.colab import files

archive = shutil.make_archive(
    "/content/trgnn-spike-diagnostic-" + run_id, "zip", root_dir=str(RUN_DIR)
)
files.download(archive)